# Design the Start of an AI Project
### A worksheet notebook on **data selection, exploitation, and experimental design**

---

Building a model is the *easy* part. Most AI projects in biology fail long before
the model — they fail at **which data you pick, how you split it, and what you
forgot to check.** This notebook walks you through those decisions for a project
*you* design.


## 1. Concept primer — what each table row is really asking

Each row of the worksheet is a trap that has sunk real projects. Here is what to
watch for. (Skim now; refer back while filling the table.)

| Row | The real question | Classic failure it prevents |
|---|---|---|
| **Biological question** | Is it *answerable with a prediction*? Phrase it as "predict/distinguish/rank X from Y." | Vague goals → no way to know if the model worked. |
| **Data modality** | What kind of signal? (images / sequence / counts / spectra) | Choosing a model that can't ingest your data. |
| **One biological sample** | What is *one row* of your dataset, biologically? | Confusing a *measurement* with an *independent sample*. |
| **Repository / source** | Where does labeled data actually exist today? | Designing around data that doesn't exist. |
| **Input format** | The concrete file/array a model eats (e.g. `224×224×3` tensor, `L×4` one-hot, gene×sample matrix). | Underestimating preprocessing. |
| **Label / target** | What are you predicting, and *who labeled it*? | No labels → no supervised model. |
| **Bias / confounder** | What varies *with* your label but isn't biology? (batch, site, camera, date) | Model learns the confounder, not the biology. |
| **Unit for train/test split** | The *independent* biological unit — split by **this**, never by measurement. | **Data leakage**: same plant/patient/plate in train *and* test → inflated accuracy. |
| **Pretrained model** | Can you stand on an existing model instead of training from scratch? | Wasting data/compute reinventing a backbone. |
| **Check before modeling** | The one sanity check that would embarrass you if skipped. | Class imbalance, label errors, leakage, resolution mismatch. |

### The two ideas most people get wrong

**1. The unit of the split.** If you photograph one leaf 20 times, those 20 images
are *not* 20 independent samples. Split by **leaf/plant/patient/genotype**, not by
image — otherwise near-duplicates leak across train and test and your accuracy is a
lie.

**2. Confounders.** If all *diseased* photos were taken on a cloudy day and all
*healthy* ones in sunshine, the model can hit 99% by learning *brightness*. Ask:
*"What else differs between my classes besides the biology?"*

## 2. Choose ONE scenario

| | Scenario | You want to… |
|---|---|---|
| **A** |  Plant disease images | study disease symptoms from field photographs |
| **B** |  Protein sequences | prioritize pathogen proteins with a particular biological function |
| **C** |  RNA-seq | distinguish two biological conditions from gene-expression profiles |
| **D** |  Hyperspectral phenotyping | detect plant stress *before* strong visible symptoms appear |

Run the next cell and pick your scenario. It prints modality-specific hints so you
don't start from a blank page.

In [ ]:
SCENARIO = "C"  #@param ["A", "B", "C", "D"]
# ↑ change to 'A', 'B', 'C', or 'D', then run this cell.

hints = {
'A': dict(title='🌿 A — Plant disease images', modality='2D RGB images (photographs)',
  sample='ONE diseased/healthy leaf or plant (not one photo!)',
  repo='PlantVillage, iNaturalist, your own field campaign, Kaggle plant-disease sets',
  fmt='RGB image tensor, e.g. resized 224x224x3, normalized',
  label='disease class (categorical) — needs an agronomist/expert to label',
  bias='camera/phone model, lighting, background soil, field site, date of capture',
  split='by PLANT or FIELD site (never by individual photo)',
  model='ImageNet-pretrained CNN/ViT (ResNet, EfficientNet, DINOv2) via timm/torchvision',
  check='class balance + leakage (same plant in train & test) + label quality'),
'B': dict(title='🧬 B — Protein sequences', modality='Amino-acid sequences (1D strings)',
  sample='ONE protein (one sequence, ideally one gene product)',
  repo='UniProt, NCBI Protein, InterPro/Pfam, PDB (for structure)',
  fmt='FASTA -> integer/one-hot encoding, or a protein-LM embedding vector',
  label='functional class / GO term / has-function-X (binary or multi-label)',
  bias='homology redundancy (near-identical sequences), taxonomic over-representation',
  split='by SEQUENCE-IDENTITY cluster (e.g. CD-HIT <30%) — NOT random rows',
  model='ESM-2 / ProtT5 / ProstT5 protein language models (HuggingFace)',
  check='sequence redundancy between train/test + label reliability (annotation source)'),
'C': dict(title='📊 C — RNA-seq (two conditions)', modality='Gene-expression count matrix',
  sample='ONE biological replicate = one RNA extraction from one organism/tissue',
  repo='NCBI GEO, SRA, ArrayExpress, Expression Atlas, your own experiment',
  fmt='genes x samples matrix (normalized: CPM/TPM/VST), samples as rows for ML',
  label='condition A vs B (binary) — from the experimental design',
  bias='BATCH EFFECT (sequencing run/lab/date), tissue, RNA quality (RIN)',
  split='by BIOLOGICAL REPLICATE / individual (never by technical replicate)',
  model='usually train small (logistic reg / RF); or pretrained scGPT/Geneformer for embeddings',
  check='batch confounding with condition + tiny n vs thousands of genes (p>>n)'),
'D': dict(title='🌈 D — Hyperspectral phenotyping', modality='Hyperspectral image/cube (many wavelength bands)',
  sample='ONE plant (or one canopy plot) imaged under controlled geometry',
  repo='public HSI phenotyping datasets, your greenhouse/field HSI campaign',
  fmt='3D cube H x W x bands (or per-pixel spectra), reflectance-calibrated',
  label='stress vs control, or a physiological measure (regression) — needs ground truth',
  bias='illumination/time-of-day, sensor drift, soil/background pixels, plant age',
  split='by PLANT and by DATE/session (avoid same plant across days leaking)',
  model='1D-CNN on spectra, or PCA/PLS-DA baseline; pretrained vision backbones need adaptation',
  check='reflectance calibration + background/soil masking + label-vs-time confound'),
}

h = hints[SCENARIO]
print('='*70)
print(h['title'])
print('='*70)
for k, lab in [('modality','Data modality'),('sample','One biological sample'),
               ('repo','Repository / source'),('fmt','Expected input format'),
               ('label','Label / target'),('bias','Biggest bias / confounder'),
               ('split','Correct unit for train/test split'),('model','Pretrained model / source'),
               ('check','Must check before modeling')]:
    print(f'\n• {lab}:\n    {h[k]}')

📊 C — RNA-seq (two conditions)

• Data modality:
    Gene-expression count matrix

• One biological sample:
    ONE biological replicate = one RNA extraction from one organism/tissue

• Repository / source:
    NCBI GEO, SRA, ArrayExpress, Expression Atlas, your own experiment

• Expected input format:
    genes x samples matrix (normalized: CPM/TPM/VST), samples as rows for ML

• Label / target:
    condition A vs B (binary) — from the experimental design

• Biggest bias / confounder:
    BATCH EFFECT (sequencing run/lab/date), tissue, RNA quality (RIN)

• Correct unit for train/test split:
    by BIOLOGICAL REPLICATE / individual (never by technical replicate)

• Pretrained model / source:
    usually train small (logistic reg / RF); or pretrained scGPT/Geneformer for embeddings

• Must check before modeling:
    batch confounding with condition + tiny n vs thousands of genes (p>>n)


## 3. Worked example (Scenario C, done for you)

Study how a *specific* answer differs from the generic hint. Notice every answer is
concrete and defends a decision.

| Question | Worked answer (RNA-seq) |
|---|---|
| Biological question | *Can we distinguish drought-stressed from well-watered rice leaves using leaf transcriptomes?* |
| Data modality | Bulk RNA-seq gene-expression counts |
| One biological sample | One RNA extraction from one plant's leaf tissue (one biological replicate) |
| Repository / source | NCBI GEO / SRA (search: *rice drought RNA-seq*) |
| Expected input format | Gene × sample count matrix → TPM/VST-normalized; samples as rows, genes as features |
| Label / target needed? | Yes — binary: `drought` vs `control`, taken from the study's design metadata |
| Biggest bias / confounder | **Batch effect**: if all drought samples were sequenced in one run and controls in another, run ≡ label |
| Correct unit for split | **Biological replicate (individual plant)** — never split technical replicates of the same plant |
| Pretrained model / source | Baseline: regularized logistic regression / random forest (n is tiny). Optional embeddings: Geneformer/scGPT |
| Must check before modeling | (1) Is batch confounded with condition? (2) p ≫ n (thousands of genes, few samples) → need strong regularization / feature selection |

**Report sentence (example):**

> *Our biological question is whether leaf transcriptomes distinguish drought-stressed
> from well-watered rice. We would obtain **bulk RNA-seq** data from **NCBI GEO/SRA**
> and split the data by **biological replicate (plant)** because technical replicates of
> the same plant would leak across train and test. Before modeling, we would check that
> **sequencing batch is not confounded with condition** and account for having far more
> genes than samples.*

# Pick another scenario and make report sentence

...............

## 4. Get REAL data — hands-on downloads

Design is theory until you touch data. Below are the main public repositories, with
direct links, then runnable cells that pull a small sample from each. **Only download
what you need** — full datasets can be huge.

###  Where the data lives

| Modality | Repository | Link |
|---|---|---|
| Plant-disease **images** | Hugging Face Datasets | https://huggingface.co/datasets |
| Plant-disease **images** | PlantVillage (original) | https://huggingface.co/datasets/mohanty/PlantVillage |
| Plant-disease **images** | PlantVillage (streamable mirror) | https://huggingface.co/datasets/geraldmc/plantvillage-tiny |
| **Apple** leaf disease | Kaggle *Plant Pathology 2021 (FGVC8)* | https://www.kaggle.com/competitions/plant-pathology-2021-fgvc8 |
| Wild observations | iNaturalist | https://www.inaturalist.org |
| **Expression** matrices | NCBI GEO | https://www.ncbi.nlm.nih.gov/geo/ |
| **Expression** matrices | EMBL-EBI Expression Atlas | https://www.ebi.ac.uk/gxa/ |
| **Expression** / arrays | ArrayExpress / BioStudies | https://www.ebi.ac.uk/biostudies/arrayexpress |
| Raw **sequencing reads** | NCBI SRA | https://www.ncbi.nlm.nih.gov/sra |
| Raw **sequencing reads** | EMBL-EBI ENA | https://www.ebi.ac.uk/ena/browser/home |
| **Protein** sequences | UniProt | https://www.uniprot.org |
| **Protein** structures | RCSB PDB | https://www.rcsb.org |
| Predicted structures | AlphaFold DB | https://alphafold.ebi.ac.uk |
